In [1]:
import pandas as pd
import numpy as np
from numpy.linalg import lstsq
from collections import defaultdict
from sklearn.metrics import accuracy_score

TICKERS = ['NVDA', 'MSFT', 'AMZN', 'META', 'GOOGL']
CATEGORIES = ['chip', 'power', 'algorithm', 'regulation', 'earnings']

# news
news = pd.read_csv('data/news_classified_final.csv')
news['Date'] = pd.to_datetime(news['Date']).dt.normalize()
news['categories'] = news['categories'].apply(eval)

# stock labels
stock_labels = {}
for ticker in TICKERS:
    df = pd.read_csv(f'data/{ticker}.csv')
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.sort_values('Date').reset_index(drop=True)
    df['next_return'] = df['Daily_Return'].shift(-1)          # 完整序列上 shift
    df['next_label']  = (df['next_return'] > 0).astype(int)
    df = df.dropna(subset=['next_return'])
    stock_labels[ticker] = df[['Date', 'Daily_Return', 'next_return', 'next_label']]

print(f"News: {len(news)}")
for t in TICKERS:
    print(f"  {t}: {len(stock_labels[t])} days")

News: 13106
  NVDA: 1253 days
  MSFT: 1253 days
  AMZN: 1253 days
  META: 1253 days
  GOOGL: 1253 days


In [2]:
def run_lp(news_df, label_df, train_start, train_end,
           test_start='2025-01-01', min_test_samples=10):
    df = pd.merge(news_df, label_df, on='Date', how='inner')
    df = df.sort_values('Date')
    df = df[df['sentiment'] != 0]
    df = df.dropna(subset=['next_return', 'next_label'])

    train = df[(df['Date'] >= train_start) & (df['Date'] < train_end)]
    test  = df[df['Date'] >= test_start].copy()

    if len(train) < 15 or len(test) < min_test_samples:
        return None

    X = np.column_stack([train['sentiment'].values,
                         train['Daily_Return'].values,
                         np.ones(len(train))])
    y = train['next_return'].values
    coef, _, _, _ = lstsq(X, y, rcond=None)
    beta = coef[0]

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)
    acc      = accuracy_score(test['next_label'], test['pred'])
    baseline = test['next_label'].mean()

    return {'beta': beta, 'acc': acc, 'baseline': baseline, 'n': len(test)}

print("run_lp ready.")

run_lp ready.


In [3]:
CATEGORIES = ['chip', 'power', 'algorithm', 'regulation', 'earnings']
BART_THRESHOLD = 0.6
FINBERT_CONF_THRESHOLD = 0.6

daily_sentiment = {}

for ticker in TICKERS:
    for cat in CATEGORIES:
        subset = news[
            (news['ticker'] == ticker) &
            (news['finbert_conf'] >= FINBERT_CONF_THRESHOLD) &
            (news['cat_scores'].apply(lambda x: eval(x).get(cat, 0) >= BART_THRESHOLD))
        ]
        if len(subset) == 0:
            continue
        daily = subset.groupby('Date')['finbert_score'].sum().reset_index()
        daily.columns = ['Date', 'sentiment']
        daily_sentiment[(ticker, cat)] = daily

print(f"BART threshold: {BART_THRESHOLD}")
print(f"FinBERT conf threshold: {FINBERT_CONF_THRESHOLD}")
print(f"Total (ticker, cat) combinations: {len(daily_sentiment)}")

BART threshold: 0.6
FinBERT conf threshold: 0.6
Total (ticker, cat) combinations: 25


In [4]:
results = []

for (ticker, cat), daily in daily_sentiment.items():
    label_df = stock_labels[ticker]
    for train_start, train_end, train_label in [
        ('2021-01-01', '2025-01-01', '2021-2024'),
        ('2023-01-01', '2025-01-01', '2023-2024')
    ]:
        res = run_lp(daily, label_df, train_start, train_end)
        if res is None:
            continue
        results.append({
            'ticker': ticker,
            'cat': cat,
            'train': train_label,
            **res
        })

results_df = pd.DataFrame(results)
results_df['beat'] = (results_df['acc'] > results_df['baseline']) & (results_df['acc'] > 0.5)

print(f"Total results: {len(results_df)}")
print(f"Beat baseline: {results_df['beat'].sum()}")

Total results: 39
Beat baseline: 15


In [5]:
all_res = results_df.sort_values(['ticker', 'cat', 'train']).copy()

cat_order = ['chip', 'power', 'algorithm', 'regulation', 'earnings']
all_res['cat'] = pd.Categorical(all_res['cat'], categories=cat_order, ordered=True)
all_res['ticker'] = pd.Categorical(all_res['ticker'], categories=TICKERS, ordered=True)
all_res = all_res.sort_values(['ticker', 'cat', 'train'])

print(f"{'Ticker':<8} {'Cat':<12} {'Train':<12} {'Beta':>10} {'Acc':>8} {'Baseline':>10} {'N':>5} {'Beat'}")
print('-' * 72)

last_ticker, last_cat = None, None
for _, row in all_res.iterrows():
    if row['ticker'] != last_ticker:
        print()
        last_ticker = row['ticker']
        last_cat = None
    if row['cat'] != last_cat:
        last_cat = row['cat']
    beat = 'Yes' if row['beat'] else 'No'
    print(f"{row['ticker']:<8} {row['cat']:<12} {row['train']:<12} {row['beta']:>+10.5f} {row['acc']:>8.1%} {row['baseline']:>10.1%} {row['n']:>5} {beat}")

Ticker   Cat          Train              Beta      Acc   Baseline     N Beat
------------------------------------------------------------------------

NVDA     chip         2021-2024      +0.00323    66.7%      54.0%    87 Yes
NVDA     chip         2023-2024      +0.00343    66.7%      54.0%    87 Yes
NVDA     power        2021-2024      +0.00120    59.3%      61.0%    59 No
NVDA     power        2023-2024      +0.00351    59.3%      61.0%    59 No
NVDA     algorithm    2021-2024      +0.00574    58.0%      58.0%    81 No
NVDA     algorithm    2023-2024      +0.00638    58.0%      58.0%    81 No
NVDA     regulation   2021-2024      +0.01189    69.6%      47.8%    23 Yes
NVDA     earnings     2021-2024      +0.00434    44.6%      55.4%    56 No
NVDA     earnings     2023-2024      +0.00443    44.6%      55.4%    56 No

MSFT     chip         2021-2024      +0.00816    68.0%      44.0%    25 Yes
MSFT     power        2021-2024      +0.00414    64.4%      53.3%    45 Yes
MSFT     power    

In [6]:
valid = results_df[
    (results_df['beat'] == True) &
    (results_df['train'] == '2021-2024')
].copy()

cat_order = ['chip', 'power', 'algorithm', 'regulation', 'earnings']
valid['cat'] = pd.Categorical(valid['cat'], categories=cat_order, ordered=True)
valid['ticker'] = pd.Categorical(valid['ticker'], categories=TICKERS, ordered=True)
valid = valid.sort_values(['ticker', 'cat'])

print(f"{'Ticker':<8} {'Cat':<12} {'Beta':>10} {'Acc':>8} {'Baseline':>10} {'N':>5}")
print('-' * 60)

last_ticker = None
for _, row in valid.iterrows():
    if row['ticker'] != last_ticker:
        print()
        last_ticker = row['ticker']
    print(f"{row['ticker']:<8} {row['cat']:<12} {row['beta']:>+10.5f} {row['acc']:>8.1%} {row['baseline']:>10.1%} {row['n']:>5}")

Ticker   Cat                Beta      Acc   Baseline     N
------------------------------------------------------------

NVDA     chip           +0.00323    66.7%      54.0%    87
NVDA     regulation     +0.01189    69.6%      47.8%    23

MSFT     chip           +0.00816    68.0%      44.0%    25
MSFT     power          +0.00414    64.4%      53.3%    45
MSFT     earnings       +0.00201    56.2%      46.9%    32

AMZN     power          +0.00756    52.6%      50.0%    38
AMZN     earnings       +0.01198    55.6%      48.1%    27

META     power          +0.01536    60.0%      47.5%    40

GOOGL    regulation     -0.00046    60.0%      50.0%    20


In [7]:
FUSION_TRAIN = '2021-2024'

# select valid combinations
valid_fusion = results_df[
    (results_df['beat'] == True) &
    (results_df['train'] == FUSION_TRAIN)
].copy()

# weighted vote per day
# step 1: collect per-stock per-day predictions
stock_day_preds = defaultdict(lambda: defaultdict(list))  # ticker -> date -> [(pred, weight)]

for _, row in valid_fusion.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    weight = row['acc']

    key = (ticker, cat)
    if key not in daily_sentiment:
        continue

    daily = daily_sentiment[key].copy()
    label_df = stock_labels[ticker]
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df[df['sentiment'] != 0]
    test = df[df['Date'] >= '2025-01-01'].copy()
    if len(test) == 0:
        continue

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)

    for _, r in test.iterrows():
        stock_day_preds[ticker][r['Date']].append((int(r['pred']), weight))

# step 2: average within each stock -> one pred per stock per day
port_day_preds = defaultdict(list)  # date -> [(pred, weight)]

for ticker, date_preds in stock_day_preds.items():
    for date, preds in date_preds.items():
        avg_pred = sum(p * w for p, w in preds) / sum(w for _, w in preds)
        final_pred = 1 if avg_pred >= 0.5 else 0
        avg_weight = sum(w for _, w in preds) / len(preds)
        port_day_preds[date].append((final_pred, avg_weight))

# step 3: weighted majority vote across stocks
# portfolio next-day label
portfolio_2025 = stock_labels
port_results = []

for date, preds in sorted(port_day_preds.items()):
    # portfolio label: majority of individual stock next_labels on this date
    labels = []
    for ticker in TICKERS:
        row = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(row) > 0:
            labels.append(int(row['next_label'].iloc[0]))
    if len(labels) == 0:
        continue
    port_label = 1 if sum(labels) >= len(labels) / 2 else 0

    total_weight = sum(w for _, w in preds)
    weight_1 = sum(w for p, w in preds if p == 1)
    final_pred = 1 if (weight_1 / total_weight) >= 0.5 else 0

    port_results.append({
        'Date': date,
        'label': port_label,
        'pred': final_pred,
        'n_stocks': len(preds),
        'correct': int(final_pred == port_label)
    })

port_df = pd.DataFrame(port_results)

acc  = port_df['correct'].mean()
base = port_df['label'].mean()

print(f"Signal days      : {len(port_df)} / 249")
print(f"Accuracy         : {acc:.1%}")
print(f"Baseline (signal): {base:.1%}")
print(f"Beat baseline    : {'Yes' if acc > base else 'No'}")
print(f"\nStock coverage per day:")
print(port_df['n_stocks'].value_counts().sort_index())

Signal days      : 166 / 249
Accuracy         : 54.2%
Baseline (signal): 51.8%
Beat baseline    : Yes

Stock coverage per day:
n_stocks
1    103
2     38
3     20
4      3
5      2
Name: count, dtype: int64


In [8]:
FOUR_TICKERS = ['NVDA', 'MSFT', 'AMZN', 'META']

valid_four = valid_fusion[valid_fusion['ticker'].isin(FOUR_TICKERS)].copy()

# step 1: per-stock per-day predictions
stock_day_preds_4 = defaultdict(lambda: defaultdict(list))

for _, row in valid_four.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    weight = row['acc']

    key = (ticker, cat)
    if key not in daily_sentiment:
        continue

    daily = daily_sentiment[key].copy()
    label_df = stock_labels[ticker]
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df[df['sentiment'] != 0]
    test = df[df['Date'] >= '2025-01-01'].copy()
    if len(test) == 0:
        continue

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)

    for _, r in test.iterrows():
        stock_day_preds_4[ticker][r['Date']].append((int(r['pred']), weight))

# step 2: average within each stock
port_day_preds_4 = defaultdict(list)

for ticker, date_preds in stock_day_preds_4.items():
    for date, preds in date_preds.items():
        avg_pred = sum(p * w for p, w in preds) / sum(w for _, w in preds)
        final_pred = 1 if avg_pred >= 0.5 else 0
        avg_weight = sum(w for _, w in preds) / len(preds)
        port_day_preds_4[date].append((final_pred, avg_weight))

# step 3: weighted majority vote
port_results_4 = []

for date, preds in sorted(port_day_preds_4.items()):
    labels = []
    for ticker in FOUR_TICKERS:
        row = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(row) > 0:
            labels.append(int(row['next_label'].iloc[0]))
    if len(labels) == 0:
        continue
    port_label = 1 if sum(labels) >= len(labels) / 2 else 0

    total_weight = sum(w for _, w in preds)
    weight_1 = sum(w for p, w in preds if p == 1)
    final_pred = 1 if (weight_1 / total_weight) >= 0.5 else 0

    port_results_4.append({
        'Date': date,
        'label': port_label,
        'pred': final_pred,
        'n_stocks': len(preds),
        'correct': int(final_pred == port_label)
    })

port_df_4 = pd.DataFrame(port_results_4)

acc  = port_df_4['correct'].mean()
base = port_df_4['label'].mean()

print(f"Signal days      : {len(port_df_4)} / 249")
print(f"Accuracy         : {acc:.1%}")
print(f"Baseline (signal): {base:.1%}")
print(f"Beat baseline    : {'Yes' if acc > base else 'No'}")
print(f"\nStock coverage per day:")
print(port_df_4['n_stocks'].value_counts().sort_index())

Signal days      : 160 / 249
Accuracy         : 55.0%
Baseline (signal): 61.3%
Beat baseline    : No

Stock coverage per day:
n_stocks
1    104
2     35
3     17
4      4
Name: count, dtype: int64


In [9]:
THREE_TICKERS = ['NVDA', 'MSFT', 'META']

valid_three = valid_fusion[valid_fusion['ticker'].isin(THREE_TICKERS)].copy()

# step 1: per-stock per-day predictions
stock_day_preds_3 = defaultdict(lambda: defaultdict(list))

for _, row in valid_three.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    weight = row['acc']

    key = (ticker, cat)
    if key not in daily_sentiment:
        continue

    daily = daily_sentiment[key].copy()
    label_df = stock_labels[ticker]
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df[df['sentiment'] != 0]
    test = df[df['Date'] >= '2025-01-01'].copy()
    if len(test) == 0:
        continue

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)

    for _, r in test.iterrows():
        stock_day_preds_3[ticker][r['Date']].append((int(r['pred']), weight))

# step 2: average within each stock
port_day_preds_3 = defaultdict(list)

for ticker, date_preds in stock_day_preds_3.items():
    for date, preds in date_preds.items():
        avg_pred = sum(p * w for p, w in preds) / sum(w for _, w in preds)
        final_pred = 1 if avg_pred >= 0.5 else 0
        avg_weight = sum(w for _, w in preds) / len(preds)
        port_day_preds_3[date].append((final_pred, avg_weight))

# step 3: weighted majority vote
port_results_3 = []

for date, preds in sorted(port_day_preds_3.items()):
    labels = []
    for ticker in THREE_TICKERS:
        row = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(row) > 0:
            labels.append(int(row['next_label'].iloc[0]))
    if len(labels) == 0:
        continue
    port_label = 1 if sum(labels) >= len(labels) / 2 else 0

    total_weight = sum(w for _, w in preds)
    weight_1 = sum(w for p, w in preds if p == 1)
    final_pred = 1 if (weight_1 / total_weight) >= 0.5 else 0

    port_results_3.append({
        'Date': date,
        'label': port_label,
        'pred': final_pred,
        'n_stocks': len(preds),
        'correct': int(final_pred == port_label)
    })

port_df_3 = pd.DataFrame(port_results_3)

acc  = port_df_3['correct'].mean()
base = port_df_3['label'].mean()

print(f"Signal days      : {len(port_df_3)} / 249")
print(f"Accuracy         : {acc:.1%}")
print(f"Baseline (signal): {base:.1%}")
print(f"Beat baseline    : {'Yes' if acc > base else 'No'}")
print(f"\nStock coverage per day:")
print(port_df_3['n_stocks'].value_counts().sort_index())

Signal days      : 140 / 249
Accuracy         : 64.3%
Baseline (signal): 52.9%
Beat baseline    : Yes

Stock coverage per day:
n_stocks
1    96
2    37
3     7
Name: count, dtype: int64


In [10]:
def bootstrap_accuracy(y_true, y_pred, n_boot=5000, ci=90):
    accs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        accs.append(accuracy_score(y_true[idx], y_pred[idx]))
    lo = (100 - ci) / 2
    hi = 100 - lo
    return np.percentile(accs, lo), np.percentile(accs, hi)

y_true = port_df_3['label'].values
y_pred = port_df_3['pred'].values

ci_lo, ci_hi = bootstrap_accuracy(y_true, y_pred, n_boot=5000)

print(f"Signal days      : {len(port_df_3)} / 249")
print(f"Accuracy         : {accuracy_score(y_true, y_pred):.1%}")
print(f"Baseline (signal): {y_true.mean():.1%}")
print(f"90% CI           : [{ci_lo:.1%}, {ci_hi:.1%}]")
print(f"Significant (CI lower > baseline): {'Yes' if ci_lo > y_true.mean() else 'No'}")

Signal days      : 140 / 249
Accuracy         : 64.3%
Baseline (signal): 52.9%
90% CI           : [57.1%, 70.7%]
Significant (CI lower > baseline): Yes


In [11]:
y_true_5 = port_df['label'].values
y_pred_5 = port_df['pred'].values

ci_lo_5, ci_hi_5 = bootstrap_accuracy(y_true_5, y_pred_5, n_boot=5000)

print(f"Signal days      : {len(port_df)} / 249")
print(f"Accuracy         : {accuracy_score(y_true_5, y_pred_5):.1%}")
print(f"Baseline (signal): {y_true_5.mean():.1%}")
print(f"90% CI           : [{ci_lo_5:.1%}, {ci_hi_5:.1%}]")
print(f"Significant (CI lower > baseline): {'Yes' if ci_lo_5 > y_true_5.mean() else 'No'}")

Signal days      : 166 / 249
Accuracy         : 54.2%
Baseline (signal): 51.8%
90% CI           : [47.6%, 60.2%]
Significant (CI lower > baseline): No


In [12]:
# large move days analysis — 3-stock portfolio (NVDA + MSFT + META)
port_2025_3 = pd.DataFrame([
    {'Date': stock_labels[ticker][stock_labels[ticker]['Date'] >= '2025-01-01']['Date'].values,
     'return': stock_labels[ticker][stock_labels[ticker]['Date'] >= '2025-01-01']['Daily_Return'].values}
    for ticker in THREE_TICKERS
])

# rebuild 3-stock portfolio daily returns
returns_3 = None
for ticker in THREE_TICKERS:
    df = stock_labels[ticker][stock_labels[ticker]['Date'] >= '2025-01-01'][['Date', 'Daily_Return']].copy()
    df = df.rename(columns={'Daily_Return': ticker})
    if returns_3 is None:
        returns_3 = df
    else:
        returns_3 = pd.merge(returns_3, df, on='Date', how='outer')

returns_3['port_return'] = returns_3[THREE_TICKERS].mean(axis=1)
returns_3 = returns_3.sort_values('Date').reset_index(drop=True)
returns_3['rolling_std'] = returns_3['port_return'].shift(1).rolling(21).std()

large_move_3 = returns_3[
    returns_3['port_return'].abs() > returns_3['rolling_std']
].dropna(subset=['rolling_std'])

print(f"Total 2025 trading days : {len(returns_3)}")
print(f"Large move days (>1 std): {len(large_move_3)}")
print(f"Large move coverage     : {len(large_move_3)/len(returns_3):.1%}")

merged_large = pd.merge(
    large_move_3[['Date', 'port_return', 'rolling_std']],
    port_df_3[['Date', 'label', 'pred']],
    on='Date', how='inner'
)

print(f"\nLarge move days with signal: {len(merged_large)}")
print(f"Coverage among large move days: {len(merged_large)/len(large_move_3):.1%}")

if len(merged_large) > 0:
    y_true_lm = merged_large['label'].values
    y_pred_lm = merged_large['pred'].values
    acc_lm  = accuracy_score(y_true_lm, y_pred_lm)
    base_lm = y_true_lm.mean()
    ci_lo_lm, ci_hi_lm = bootstrap_accuracy(y_true_lm, y_pred_lm, n_boot=5000)

    print(f"\n=== Performance on Large Move Days ===")
    print(f"Accuracy         : {acc_lm:.1%}")
    print(f"Baseline         : {base_lm:.1%}")
    print(f"90% CI           : [{ci_lo_lm:.1%}, {ci_hi_lm:.1%}]")
    print(f"Significant (CI lower > baseline): {'Yes' if ci_lo_lm > base_lm else 'No'}")
    print(f"\n=== Summary ===")
    print(f"All signal days (n={len(port_df_3)}): Accuracy {accuracy_score(y_true, y_pred):.1%}, Baseline {y_true.mean():.1%}")
    print(f"Large move days (n={len(merged_large)}): Accuracy {acc_lm:.1%}, Baseline {base_lm:.1%}")

Total 2025 trading days : 248
Large move days (>1 std): 66
Large move coverage     : 26.6%

Large move days with signal: 40
Coverage among large move days: 60.6%

=== Performance on Large Move Days ===
Accuracy         : 82.5%
Baseline         : 55.0%
90% CI           : [72.5%, 92.5%]
Significant (CI lower > baseline): Yes

=== Summary ===
All signal days (n=140): Accuracy 64.3%, Baseline 52.9%
Large move days (n=40): Accuracy 82.5%, Baseline 55.0%


In [13]:
# force predict all large move days — no signal days default to 1 (up)
large_move_dates = set(large_move_3['Date'].values)

full_large = []
for _, row in large_move_3.iterrows():
    date = row['Date']
    # check if model has signal on this date
    signal_row = port_df_3[port_df_3['Date'] == date]
    
    # get true label
    labels = []
    for ticker in THREE_TICKERS:
        r = stock_labels[ticker][stock_labels[ticker]['Date'] == date]
        if len(r) > 0:
            labels.append(int(r['next_label'].iloc[0]))
    if len(labels) == 0:
        continue
    port_label = 1 if sum(labels) >= len(labels) / 2 else 0
    
    if len(signal_row) > 0:
        pred = int(signal_row['pred'].iloc[0])
        has_signal = True
    else:
        pred = 1  # default to up
        has_signal = False
    
    full_large.append({
        'Date': date,
        'label': port_label,
        'pred': pred,
        'has_signal': has_signal,
        'correct': int(pred == port_label)
    })

full_large_df = pd.DataFrame(full_large)

y_true_fl = full_large_df['label'].values
y_pred_fl = full_large_df['pred'].values
acc_fl  = accuracy_score(y_true_fl, y_pred_fl)
base_fl = y_true_fl.mean()
ci_lo_fl, ci_hi_fl = bootstrap_accuracy(y_true_fl, y_pred_fl, n_boot=5000)

print(f"=== Large Move Days Full Coverage (default: up) ===")
print(f"Total large move days   : {len(full_large_df)}")
print(f"  - with signal         : {full_large_df['has_signal'].sum()}")
print(f"  - default up          : {(~full_large_df['has_signal']).sum()}")
print(f"Accuracy                : {acc_fl:.1%}")
print(f"Baseline                : {base_fl:.1%}")
print(f"90% CI                  : [{ci_lo_fl:.1%}, {ci_hi_fl:.1%}]")
print(f"Significant             : {'Yes' if ci_lo_fl > base_fl else 'No'}")

=== Large Move Days Full Coverage (default: up) ===
Total large move days   : 66
  - with signal         : 40
  - default up          : 26
Accuracy                : 71.2%
Baseline                : 54.5%
90% CI                  : [62.1%, 80.3%]
Significant             : Yes
